# Module 05: Hugging Face Ecosystem & BERT Fine-Tuning Mastery
## Master Guide: Pre-Trained Transformers, Tokenizers, and Transfer Learning for NLP

In 2018, Google AI introduced **BERT** (*Bidirectional Encoder Representations from Transformers*), marking the dawn of the modern pre-trained Transformer era. Instead of training text models from scratch on small datasets, NLP adopted computer vision's winning recipe: **Pre-train on massive text corpora, then fine-tune on target tasks with minimal compute!**

Today, the **Hugging Face `transformers` library** is the global industry standard for downloading, customizing, training, and deploying transformer models.

---

### 🎯 Key Learning Objectives in this Module:
1. **The Pre-Training & Fine-Tuning Paradigm**
   - The 3 Architectural Branches: Encoder-Only (BERT), Decoder-Only (GPT), Encoder-Decoder (T5)
   - BERT's Pre-Training Objectives: Masked Language Modeling (MLM) and Next Sentence Prediction (NSP)
2. **Hugging Face `transformers` Architecture**
   - `pipeline()` API for zero-effort, high-performance inference
   - `AutoTokenizer`: WordPiece, Byte-Pair Encoding (BPE), Special Tokens (`[CLS]`, `[SEP]`, `[PAD]`)
   - `AutoModelForSequenceClassification` and Model Heads
3. **Hands-On End-to-End Fine-Tuning Project**:
   - Loading pre-trained DistilBERT
   - Tokenization & PyTorch `Dataset` with attention masks
   - Training loop with `AdamW` optimizer and learning rate warmup scheduler
   - Evaluating Precision, Recall, F1-Score, and Confusion Matrix
   - Exporting the fine-tuned model and tokenizer for real-world production serving
4. **Self-Assessment Practice Exercises & Detailed Solutions**


---
## 1. The Pre-Training & Fine-Tuning Paradigm

Before Transformers, word vectors like Word2Vec and GloVe were **static**: the word *"bank"* received the exact same vector whether in *"river bank"* or *"investment bank"*.

BERT solved this by computing **dynamic contextual embeddings** where a word's representation depends on all surrounding words in both directions.

### The 3 Architectural Archetypes of Transformers:

```text
               Transformer Architecture
                         |
      +------------------+------------------+
      |                  |                  |
Encoder-Only       Decoder-Only       Encoder-Decoder
(Autoencoding)     (Autoregressive)   (Sequence-to-Sequence)
      |                  |                  |
BERT, RoBERTa      GPT-4, Llama       T5, BART
DeBERTa, DistilBERT Mistral, Claude   MarianMT
      |                  |                  |
- Bidirectional    - Left-to-right    - Encoder reads input,
  context            causal attention   decoder writes output
- Best for:        - Best for:        - Best for:
  * Classification   * Text generation  * Translation
  * NER, POS         * Code generation  * Summarization
  * Extractive QA    * Creative writing * Paraphrasing
```

### BERT's Dual Pre-Training Objectives:
1. **Masked Language Modeling (MLM)**:
   - 15% of all tokens in a sentence are selected for masking.
   - Of these: 80% are replaced with `[MASK]`, 10% are replaced with a random token, and 10% are kept unchanged.
   - The model must predict the original word using bidirectional context!
2. **Next Sentence Prediction (NSP)**:
   - The model is fed Sentence A and Sentence B, separated by `[SEP]`.
   - It predicts a binary flag: Is Sentence B the actual subsequent sentence, or a random sentence from the corpus?


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import numpy as np
import matplotlib.pyplot as plt

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch using device: {device}")


---
## 2. Fast Prototyping with Hugging Face `pipeline`

The `pipeline()` function encapsulates the entire NLP workflow:
$$\text{Raw Text} \xrightarrow{\text{Tokenizer}} \text{Tensors} \xrightarrow{\text{Model}} \text{Logits} \xrightarrow{\text{Post-Processing}} \text{Human Predictions}$$

Let's test Sentiment Analysis and Zero-Shot Classification in 3 lines of code!


In [ ]:
# 1. Zero-shot Sentiment Analysis Pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

reviews = [
    "The step-by-step mathematical explanations made deep learning so clear and intuitive!",
    "I was frustrated by the terrible documentation and broken examples."
]

results = sentiment_pipeline(reviews)
for text, res in zip(reviews, results):
    print(f"Review: '{text}'")
    print(f"  -> Label: {res['label']}, Score: {res['score']:.4f}\n")


In [ ]:
# 2. Zero-Shot Topic Classification
# Classify text into categories WITHOUT training on them!
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

sample_article = "The central bank announced a 25 basis point reduction in interest rates to spur economic growth."
candidate_topics = ["Sports", "Finance & Economics", "Healthcare", "Aerospace Engineering"]

zero_shot_res = zero_shot_classifier(sample_article, candidate_labels=candidate_topics)
print(f"Article: '{sample_article}'")
for label, score in zip(zero_shot_res['labels'], zero_shot_res['scores']):
    print(f"  [{label}]: {score * 100:.2f}%")


---
## 3. Deep Dive into Tokenizers: WordPiece, Subwords & Tensors

Modern transformers do **not** split by white spaces or characters. They use **Subword Tokenization** (WordPiece in BERT, BPE in GPT/RoBERTa).
- Frequently occurring words remain intact: `"apple"` $\to$ `["apple"]`.
- Rare or compound words are broken into subword pieces: `"unbelievable"` $\to$ `["un", "##believ", "##able"]`.
- This eliminates the out-of-vocabulary (`<UNK>`) problem completely!

### Key Tensors Output by `tokenizer()`:
1. `input_ids`: The numerical indices of each token from the model's vocabulary.
2. `attention_mask`: 1 for real tokens, 0 for padding tokens (tells the attention layer to ignore padding).
3. `token_type_ids`: 0 for Sentence A, 1 for Sentence B (used for sentence-pair tasks).


In [ ]:
# Load the DistilBERT Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_sentence = "Self-attention transforms NLP models extraordinarily!"
encoded = tokenizer(
    sample_sentence,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print(f"Original Text: '{sample_sentence}'")
print(f"Tokens: {tokens}")
print(f"Input IDs: {encoded['input_ids'][0].tolist()}")
print(f"Attention Mask: {encoded['attention_mask'][0].tolist()}")


---
## 4. Fine-Tuning DistilBERT on a Custom Text Classification Dataset

Now, let's perform full fine-tuning! We will take `distilbert-base-uncased` (which has 66M pre-trained parameters) and fine-tune it for domain-specific technical review sentiment classification.

We will write a clean, native PyTorch training loop using:
- `AutoModelForSequenceClassification`
- PyTorch `Dataset` & `DataLoader`
- `torch.optim.AdamW` with weight decay
- Learning rate warmup scheduler
- Evaluation metrics calculation


In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Domain-specific training data (e.g. AI Research Code Reviews)
dataset_samples = [
    ("The gradient calculation is mathematically sound and runs fast.", 1),
    ("Clean implementation of multi-head attention with clear docstrings.", 1),
    ("Superb optimization, memory footprint dropped by 60 percent.", 1),
    ("Extremely well structured notebook, learned so much from the derivations.", 1),
    ("The PyTorch module is bug-free and achieves state of the art results.", 1),
    ("The loss exploded to NaN because of missing gradient clipping.", 0),
    ("Horrible runtime performance, severe memory leak during backprop.", 0),
    ("Completely broken code, shapes do not match in matrix multiplication.", 0),
    ("Poor explanations, unreadable variable names and confusing math.", 0),
    ("Fails on edge cases and crashes when sequence length is greater than 10.", 0)
]

validation_samples = [
    ("Great explanation of the attention weights matrix and eigenvalues.", 1),
    ("The model converges very smoothly with AdamW optimizer.", 1),
    ("Broken imports and deprecated functions make this unusable.", 0),
    ("The code throws an out of memory CUDA error on batch size 4.", 0)
]

class HFTextDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=24):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        text, label = self.data[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

train_dataset = HFTextDataset(dataset_samples, tokenizer)
val_dataset = HFTextDataset(validation_samples, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

print(f"Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}")


In [ ]:
# Load Pre-trained Model with Classification Head
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model.to(device)

# Configure Optimizer with AdamW (weight decay regularizes pre-trained weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)

# Linear Warmup Learning Rate Scheduler
total_epochs = 4
total_steps = len(train_loader) * total_epochs
from transformers import get_linear_schedule_with_warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)


In [ ]:
# Fine-Tuning Loop
print("=== Starting Fine-Tuning ===")
for epoch in range(1, total_epochs + 1):
    model.train()
    total_train_loss = 0.0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation step
    model.eval()
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_targets, val_preds)
    print(f"Epoch {epoch}/{total_epochs} | Train Loss: {avg_train_loss:.4f} | Val Accuracy: {val_acc * 100:.1f}%")


In [ ]:
# Save the Fine-Tuned Model and Tokenizer to Disk
save_directory = "./fine_tuned_distilbert_nlp101"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model and Tokenizer successfully exported to: {save_directory}")

# Reload model from disk and test in an end-to-end inference function
reloaded_model = AutoModelForSequenceClassification.from_pretrained(save_directory).to(device)
reloaded_tokenizer = AutoTokenizer.from_pretrained(save_directory)

def analyze_review(text):
    reloaded_model.eval()
    inputs = reloaded_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = reloaded_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        prediction = np.argmax(probs)
    
    labels = {0: "NEGATIVE (Buggy / Poor Code)", 1: "POSITIVE (High Quality Code)"}
    return labels[prediction], probs[prediction]

test_review = "This PyTorch model implementation is clean, fast, and highly accurate."
label, conf = analyze_review(test_review)
print(f"\nTest Review: '{test_review}'")
print(f"Prediction: {label} (Confidence: {conf*100:.2f}%)")


---
## 5. Self-Assessment Practice Exercises & Detailed Solutions

### Exercise 1: Why AdamW instead of standard Adam?
What is the difference between standard $L_2$ regularization in Adam vs decoupled Weight Decay in AdamW (Loshchilov & Hutter, 2017), and why is AdamW crucial for pre-trained Transformers?

---
### Exercise 2: Choosing Model Archetype
For each of the following tasks, state whether you should choose an **Encoder-Only**, **Decoder-Only**, or **Encoder-Decoder** model, and name an example model:
1. Extracting company names and dollar amounts from financial contracts (Named Entity Recognition).
2. Translating technical documentation from English to German.
3. Writing a Python script from an open-ended conversational instruction.

---
### Solutions:
<details>
<summary><b>Click to expand Solution 1</b></summary>

In standard SGD, $L_2$ regularization is mathematically equivalent to weight decay.  
However, in adaptive gradient methods like Adam, standard $L_2$ regularization adds $\lambda w$ directly to the gradient $g_t$, which then gets divided by the running estimate of the second moment $\sqrt{v_t} + \epsilon$. As a result:
- Weights with large historical gradients receive **less** weight decay!
- Weights with small historical gradients receive **more** weight decay!
**AdamW** decouples the weight decay update so it is applied directly to the weights:
$$w_{t+1} = w_t - \eta \lambda w_t - \frac{\eta}{\sqrt{v_t} + \epsilon} m_t$$
This ensures uniform regularization across all parameters, which dramatically improves generalization when fine-tuning deep pre-trained Transformers.
</details>

<details>
<summary><b>Click to expand Solution 2</b></summary>

1. **Named Entity Recognition (NER)**: **Encoder-Only** (e.g., BERT, RoBERTa, DeBERTa). Requires rich bidirectional context for every token to accurately identify entity boundaries.
2. **English-to-German Translation**: **Encoder-Decoder** (e.g., T5, MarianMT). The encoder ingests the entire source English sentence, and the decoder autoregressively generates the target German tokens.
3. **Conversational Code Generation**: **Decoder-Only** (e.g., GPT-4, Llama 3, StarCoder, Mistral). Autoregressive generation excels at dynamic token prediction and instruction-following.
</details>


---
## 6. Summary & Next Steps
1. **Pre-training and fine-tuning** enables massive sample efficiency: the model already understands English grammar, syntax, and vocabulary before seeing a single domain sample.
2. **Subword tokenization** solves the OOV dilemma by decomposing rare words into morphological sub-tokens.
3. The **`transformers` library** standardizes loading pre-trained weights, tokenizers, and task-specific classification heads.
4. In **Module 06**, we enter the era of **Generative AI, Large Language Models (LLMs), Prompt Engineering, Vector Databases, and Retrieval-Augmented Generation (RAG)**!
